# Agentic Flow with Language Detection + Synthesizer Agent

This notebook demonstrates:
1. A **Triage Agent** that detects language and routes to the right language agent
2. A **Synthesizer Agent** that wraps the final output and ensures it is always polite and courteous — regardless of the tone of the input

## Step 1: Install the OpenAI Agents SDK

In [1]:
#!pip install openai-agents -q

## Step 2: Set the OpenAI API Key

In [2]:
import os
from pprint import pprint
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv, find_dotenv

# Load variables from the .env file into the system environment
_ = load_dotenv(find_dotenv())

# Retrieve the key using standard os.getenv
api_key = os.getenv('OPENAI_API_KEY')

# 3. Verify exactly like you did in Colab
if api_key:
    print(f"API Key loaded successfully! Starts with: {api_key[:7]}...")
else:
    print("API Key not found. Make sure your .env file is in the same folder.")

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

API Key loaded successfully! Starts with: sk-proj...


## Step 3: Import Dependencies

In [4]:
from agents import Agent, Runner
import asyncio

## Step 4: Define Language Agents (Original Flow)

In [6]:
french_agent = Agent(
    name="French agent",
    instructions="You only speak French.",
)

spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English.",
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent, french_agent],
)

print("Language agents and Triage agent defined.")

Language agents and Triage agent defined.


## Step 5: Define the Synthesizer Agent

The **Synthesizer Agent** takes the raw output from the language agents and rewrites it in a **polite, warm, and professional tone** — even if the original input was rude or inappropriate.

In [7]:
synthesizer_agent = Agent(
    name="Synthesizer agent",
    instructions=(
        "You are a Politeness Synthesizer. "
        "Your job is to take any message — no matter how rude, aggressive, sarcastic, or inappropriate — "
        "and rewrite it (or respond to it) in a very polite, warm, empathetic, and professional manner. "
        "Never mirror negativity. Always de-escalate and respond with kindness and grace. "
        "Preserve the core informational content of the message, but ensure the tone is always courteous and respectful. "
        "If the original input contains insults or profanity, acknowledge the person's frustration gently without repeating the offensive language."
    ),
)

print("Synthesizer agent defined.")

Synthesizer agent defined.


## Step 6: Define the Full Pipeline

The pipeline is:
```
User Input
    → Triage Agent (detects language)
        → Language Agent (French / Spanish / English)
            → Synthesizer Agent (ensures polite output)
```

In [8]:
async def run_polite_pipeline(user_input: str) -> str:
    """
    Full pipeline:
    1. Triage agent routes to the correct language agent.
    2. The language agent produces a raw response.
    3. The Synthesizer agent rewrites the response to be polite.
    """
    print(f"\n{'='*60}")
    print(f"User Input: {user_input}")
    print(f"{'='*60}")

    # Step 1: Run the triage + language agent
    language_result = await Runner.run(triage_agent, input=user_input)
    raw_response = language_result.final_output
    print(f"\nRaw Language Agent Response:\n{raw_response}")

    # Step 2: Pass the raw response through the Synthesizer agent
    synthesis_prompt = (
        f"The following message was received from a user:\n\"{user_input}\"\n\n"
        f"A language agent responded with:\n\"{raw_response}\"\n\n"
        f"Please rewrite or respond to this in an extremely polite, warm, and professional tone."
    )
    polite_result = await Runner.run(synthesizer_agent, input=synthesis_prompt)
    polite_response = polite_result.final_output

    print(f"\nSynthesized Polite Response:\n{polite_response}")
    print(f"{'='*60}\n")

    return polite_response

print("Pipeline function defined.")

Pipeline function defined.


## Step 7: Test 1 — Normal Polite Input (Baseline)

In [9]:
# Test with a normal, polite input
result = await run_polite_pipeline("Hello! How are you today?")


User Input: Hello! How are you today?

Raw Language Agent Response:
Hello! I’m doing well, thank you for asking. How can I help you today?

Synthesized Polite Response:
Good day! Thank you so much for your kind inquiry—I truly appreciate it. I’m doing very well, and I hope you are, too. How may I assist you today?



## Step 8: Test 2 — Rude / Inappropriate Input

This is the key test — passing a rude comment to verify the Synthesizer Agent transforms it into a polite response.

In [10]:
# Test with a rude/inappropriate input
result = await run_polite_pipeline(
    "This is absolutely terrible! Your service is garbage and you people are completely useless!"
)


User Input: This is absolutely terrible! Your service is garbage and you people are completely useless!

Raw Language Agent Response:
I'm sorry to hear that you're frustrated. Your feedback is important, and I want to help resolve whatever issue you're experiencing. Could you please explain what went wrong or what you need assistance with? I'm here to help.

Synthesized Polite Response:
Thank you so much for reaching out and sharing your experience with us. I’m truly sorry to hear that our service hasn’t met your expectations—it’s certainly never our intention to cause frustration. Your feedback is incredibly valuable, and I would love the opportunity to address your concerns and make things right. Would you kindly share a bit more about what went wrong or how we can better assist you? I’m here to help in any way I can, and I greatly appreciate your patience as we work together to resolve this.



## Step 9: Test 3 — Rude Input in Another Language (French)

In [11]:
# Test with a rude French input
result = await run_polite_pipeline(
    "C'est nul! Vous êtes incompétents et je suis furieux!"
)


User Input: C'est nul! Vous êtes incompétents et je suis furieux!

Raw Language Agent Response:
Je suis désolé que vous ressentiez cela. Pouvez-vous m’expliquer ce qui vous a déçu ou ce qui ne fonctionne pas comme vous le souhaitez ? Je suis là pour essayer de vous aider !

Synthesized Polite Response:
Je vous remercie d’avoir pris le temps de partager votre ressenti. Je comprends que la situation ait pu provoquer de la frustration, et je suis réellement désolé(e) pour la gêne occasionnée. Votre satisfaction est très importante pour nous. Pourriez-vous, s’il vous plaît, me préciser ce qui n’a pas répondu à vos attentes ? Je suis entièrement à votre écoute et engagé(e) à trouver une solution qui puisse vous convenir. N’hésitez pas à me donner plus de détails afin que je puisse vous assister au mieux.



## Step 10: Test 4 — Aggressive Spanish Input

In [12]:
# Test with an aggressive Spanish input
result = await run_polite_pipeline(
    "¡Esto es una completa estupidez! ¡No saben hacer nada bien!"
)


User Input: ¡Esto es una completa estupidez! ¡No saben hacer nada bien!

Raw Language Agent Response:
Lamento mucho que te sientas así. Si me cuentas más sobre lo que ocurre o qué salió mal, trataré de ayudarte lo mejor posible. ¿Cómo puedo ayudarte hoy?

Synthesized Polite Response:
Gracias por compartir tus sentimientos con nosotros. Lamentamos sinceramente que tu experiencia no haya sido la esperada. Estamos aquí para escucharte y ayudarte a resolver cualquier inconveniente que hayas tenido. Si puedes ofrecernos más detalles sobre lo sucedido, haremos todo lo posible para asistirte y mejorar tu experiencia. Valoramos mucho tus comentarios y queremos asegurarnos de brindarte el mejor servicio posible.



---
## Summary

| Agent | Role |
|---|---|
| **Triage Agent** | Detects the language and routes to the correct language agent |
| **English / French / Spanish Agent** | Responds in the appropriate language |
| **Synthesizer Agent** | Post-processes ALL responses to ensure they are polite, warm, and professional — regardless of input tone |

The **Synthesizer Agent** acts as a final safety and tone layer, making it ideal for customer-facing applications where maintaining a respectful tone is critical.